# STimage-1K4M → HEST Conversion Pipeline
**Project:** Predicting expression from H&E — Campbell Lab, Matthew Perrone (May 2026)

**Purpose:** Expand the HEST training dataset by converting STimage-1K4M Visium samples into HEST format, correcting the spinal cord/brain tissue bias in HEST-1K and adding human cancer tissue diversity.

| | HEST-1K (before) | This pipeline adds |
|---|---|---|
| Visium samples | 172 | +179 from 1K4M |
| Human cancer tissue types | 13 | +10 new |
| Visium patches | ~528K | +1,832K |
| **Total patches (all techs)** | **777K** | **→ 2,609K** |

**Data flow:**
```
HuggingFace  jiawennnn/STimage-1K4M
      ↓  coord CSV + count CSV + image (per sample)
  [this pipeline]
      ↓  AnnData · pyramidal TIFF · patches
HuggingFace  MahmoodLab/hest  (new version)
```

**Expected output:** ~179 samples · ~1.83M patches · ~100–155 GB total

> **Kernel:** use the `hest` conda environment — `/opt/anaconda3/envs/hest/bin/python`

---
### Stages
1. **Scope filter + Overlap detection** — filter 1K4M to Visium/human/cancer → 6-feature score against HEST-1K → `stimage_novel.csv`
2. **Conversion** — per novel sample: download → AnnData → HESTData → segment → save → patch
3. **Upload** — stream completed samples to HuggingFace `MahmoodLab/hest`

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────
# pip install hest huggingface_hub scanpy scipy pillow openslide-python
import os, re, json, shutil
from pathlib import Path
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
from PIL import Image
from huggingface_hub import hf_hub_download, HfApi
from hest import HESTData

# ── Paths ─────────────────────────────────────────────────────────────────
# Notebook lives in:  .../GitHub/Hest Assembling Pipeline/
# STimage-1K4M is in: .../GitHub/STimage-1K4M/
STIMAGE_META    = Path('../STimage-1K4M/meta/meta_all_gene02122025.csv')
HEST_META_HF    = 'hf://datasets/MahmoodLab/hest/HEST_v1_3_0.csv'
HF_STIMAGE_REPO = 'jiawennnn/STimage-1K4M'
HF_HEST_REPO    = 'MahmoodLab/hest'

PIPELINE_OUT = Path('pipeline_output')   # overlap-detection CSVs
HEST_OUT     = Path('hest_output')       # converted HEST samples
SCRATCH      = Path('scratch')           # temp download space

for d in [PIPELINE_OUT, HEST_OUT, SCRATCH]:
    d.mkdir(exist_ok=True)

# ── Conversion config ─────────────────────────────────────────────────────
PATCH_SIZE            = 256    # px  — matches UNI foundation model input
TARGET_PIX_SIZE       = 0.5    # µm/px
VISIUM_SPOT_RADIUS_UM = 27.5   # Visium spot radius (55 µm diameter)
TITLE_THRESHOLD       = 0.6    # fuzzy-match cutoff for paper titles

# Sanity check
assert STIMAGE_META.exists(), f'Not found: {STIMAGE_META.resolve()}'
print(f'STimage meta  : {STIMAGE_META.resolve()}')
print('Environment ready.')

---
## Part 1 — Scope Filter + Overlap Detection

**Goal:** From 1,149 STimage-1K4M slides, isolate candidates that are:
- `tech == "Visium"` (not STv1 or VisiumHD)
- `species == "human"`
- `involve_cancer == True`

Then score those candidates against HEST-1K with 5 features to identify truly **novel** samples not already curated.

| # | Feature | Match type |
|---|---|---|
| 1 | GSM accession ID (slide name) | Exact — sample level |
| 2 | GSE study ID (slide name) | Exact — study level |
| 3 | PubMed ID (`pmid` column) | Exact — paper level |
| 4 | Paper title | Fuzzy SequenceMatcher ≥ 0.6 |
| 5 | Species + tissue + spot count ±10% | Composite fingerprint |

Technology is not scored — all candidates are already Visium by construction.

Confidence ≥ 1.0 → **DEFINITE** (skip) · ≥ 0.6 → **LIKELY** (skip) · ≥ 0.3 → **POSSIBLE** (manual review) · < 0.3 → **NOVEL** (convert)

In [ ]:
# ── Load metadata ─────────────────────────────────────────────────────────
st_meta   = pd.read_csv(STIMAGE_META)
hest_meta = pd.read_csv(HEST_META_HF)

print(f'STimage-1K4M total : {len(st_meta):,} samples')
print(f'HEST-1K total      : {len(hest_meta):,} samples')
print(f'\nSTimage-1K4M tech breakdown:\n{st_meta.tech.value_counts().to_string()}')
print(f'\nInvolve cancer:  {st_meta.involve_cancer.value_counts().to_dict()}')

# ── Scope filter — applied BEFORE overlap scoring ─────────────────────────
# Only target: Visium + human + cancer (involve_cancer is bool in meta CSV)
candidates = st_meta[
    (st_meta['tech'] == 'Visium') &
    (st_meta['species'].str.lower() == 'human') &
    (st_meta['involve_cancer'] == True)
].copy().reset_index(drop=True)

print(f'\nAfter scope filter (Visium + human + cancer): {len(candidates):,} samples')
print(f'Tissue type breakdown:\n{candidates["tissue"].value_counts().to_string()}')

In [ ]:
# ── Features 1 & 2: GSM and GSE accession IDs ────────────────────────────
# 1K4M slide names encode GEO accessions: e.g. "GSE144239_GSM4284316"
def extract_id(pattern, text):
    m = re.search(pattern, str(text))
    return m.group(1) if m else None

candidates['gsm_id'] = candidates['slide'].apply(lambda x: extract_id(r'(GSM\d+)', x))
candidates['gse_id'] = candidates['slide'].apply(lambda x: extract_id(r'(GSE\d+)', x))

hest_meta['gsm_id'] = hest_meta['download_page_link1'].apply(lambda x: extract_id(r'acc=(GSM\d+)', x))
hest_meta['gse_id'] = (
    hest_meta['download_page_link1'].apply(lambda x: extract_id(r'acc=(GSE\d+)', x))
    .fillna(hest_meta['study_link'].apply(lambda x: extract_id(r'acc=(GSE\d+)', x)))
)

hest_gsm_set = set(hest_meta['gsm_id'].dropna())
hest_gse_set = set(hest_meta['gse_id'].dropna())

candidates['feat1_gsm'] = candidates['gsm_id'].isin(hest_gsm_set)
candidates['feat2_gse'] = candidates['gse_id'].isin(hest_gse_set)

print(f'Feat 1 — GSM exact match : {candidates.feat1_gsm.sum()} samples (definitive overlap)')
print(f'Feat 2 — GSE study match : {candidates.feat2_gse.sum()} samples')

In [ ]:
# ── Feature 3: PubMed ID ──────────────────────────────────────────────────
def pmid_set(val):
    if pd.isna(val): return set()
    return {p.strip() for p in str(val).split(',') if p.strip().isdigit()}

candidates['pmid_set'] = candidates['pmid'].apply(pmid_set)
hest_meta['pmid'] = hest_meta['study_link'].apply(
    lambda x: extract_id(r'pubmed\.ncbi\.nlm\.nih\.gov/(\d+)', x)
)
hest_pmid_set = set(hest_meta['pmid'].dropna())

candidates['feat3_pmid'] = candidates['pmid_set'].apply(lambda s: bool(s & hest_pmid_set))
shared = {p for s in candidates[candidates.feat3_pmid]['pmid_set'] for p in s} & hest_pmid_set
print(f'Feat 3 — PMID match : {candidates.feat3_pmid.sum()} samples  |  shared PMIDs: {sorted(shared)}')

In [ ]:
# ── Feature 4: Paper title fuzzy match ───────────────────────────────────
def normalise(text):
    if pd.isna(text): return ''
    return re.sub(r'[^a-z0-9 ]', '', str(text).lower())

hest_titles_norm = hest_meta['dataset_title'].apply(normalise).tolist()

def best_title_score(raw):
    s = normalise(raw)
    if not s: return 0.0
    return round(max(SequenceMatcher(None, s[:200], h[:200]).ratio() for h in hest_titles_norm), 3)

print(f'Computing title similarity for {len(candidates)} candidates (~1 min)...')
candidates['feat4_title_score'] = candidates['title'].apply(best_title_score)
candidates['feat4_title']       = candidates['feat4_title_score'] >= TITLE_THRESHOLD
print(f'Feat 4 — Title fuzzy (≥{TITLE_THRESHOLD}) : {candidates.feat4_title.sum()} samples')

In [ ]:
# ── Feature 5: Composite fingerprint (species + tissue + spot count ±10%) ─
# Note: technology compatibility is not scored — all candidates are already
# Visium (scope-filtered), and HEST-1K contains Visium, so it would be True
# for every candidate and add nothing discriminating.
candidates['tissue_norm'] = candidates['tissue'].str.lower().str.strip()
hest_meta['tissue_norm']  = hest_meta['tissue'].str.lower().str.strip()

SPOT_TOL = 0.10
fp_lookup: dict = {}
for _, row in hest_meta.iterrows():
    key = (str(row.get('species', '')).lower(), row.get('tissue_norm', ''))
    if pd.notna(row.get('spots_under_tissue')):
        fp_lookup.setdefault(key, []).append(int(row['spots_under_tissue']))

def fingerprint_match(row):
    key = ('homo sapiens', row['tissue_norm'])
    if key not in fp_lookup or pd.isna(row['spot_num']): return False
    st_spots = int(row['spot_num'])
    return any(abs(st_spots - h) / max(st_spots, 1) <= SPOT_TOL for h in fp_lookup[key])

candidates['feat5_fingerprint'] = candidates.apply(fingerprint_match, axis=1)
print(f'Feat 5 — Fingerprint match: {candidates.feat5_fingerprint.sum()} samples')

In [ ]:
# ── Confidence scoring ────────────────────────────────────────────────────
WEIGHTS = {
    'feat1_gsm'         : 1.00,   # exact sample ID — definitive
    'feat2_gse'         : 0.60,   # same study
    'feat3_pmid'        : 0.60,   # same paper
    'feat4_title'       : 0.40,   # fuzzy title
    'feat5_fingerprint' : 0.30,   # species + tissue + spot count ±10%
}

candidates['confidence'] = sum(
    candidates[f].astype(float) * w for f, w in WEIGHTS.items()
)

def overlap_label(score):
    if score >= 1.00: return 'DEFINITE'    # skip — confirmed duplicate
    if score >= 0.60: return 'LIKELY'      # skip — very probably in HEST-1K
    if score >= 0.30: return 'POSSIBLE'    # manual review
    return 'NOVEL'                          # convert to HEST

candidates['overlap_label'] = candidates['confidence'].apply(overlap_label)

print('=== Overlap label counts (Visium + human + cancer candidates) ===')
print(candidates['overlap_label'].value_counts().to_string())
print(f'\nNOVEL samples ready for conversion: {(candidates.overlap_label == "NOVEL").sum()}')

In [ ]:
# ── Save outputs ──────────────────────────────────────────────────────────
feat_cols = [
    'slide', 'gsm_id', 'gse_id', 'pmid', 'tech', 'tissue', 'species',
    'spot_num', 'gene_num', 'involve_cancer',
    'feat1_gsm', 'feat2_gse', 'feat3_pmid', 'feat4_title', 'feat4_title_score',
    'feat5_tech', 'feat6_fingerprint', 'confidence', 'overlap_label',
]

candidates[feat_cols].to_csv(PIPELINE_OUT / 'stimage_overlap_scored.csv', index=False)

novel   = candidates[candidates['overlap_label'] == 'NOVEL'].copy()
review  = candidates[candidates['overlap_label'] == 'POSSIBLE'].copy()
overlap = candidates[candidates['overlap_label'].isin(['DEFINITE', 'LIKELY'])].copy()

novel.to_csv(PIPELINE_OUT   / 'stimage_novel.csv',             index=False)
review.to_csv(PIPELINE_OUT  / 'stimage_manual_review.csv',     index=False)
overlap.to_csv(PIPELINE_OUT / 'stimage_confirmed_overlap.csv', index=False)

print(f'Saved to {PIPELINE_OUT}/')
print(f'  stimage_novel.csv             — {len(novel):>4}  samples  →  convert')
print(f'  stimage_manual_review.csv     — {len(review):>4}  samples  →  review')
print(f'  stimage_confirmed_overlap.csv — {len(overlap):>4}  samples  →  skip')

### Manual review guide

For each row in `stimage_manual_review.csv` (POSSIBLE label):
1. **GSE number** → search https://www.ncbi.nlm.nih.gov/geo/
2. **Title** → compare against HEST `dataset_title` in `stimage_overlap_scored.csv`
3. **PMID** → cross-check at https://pubmed.ncbi.nlm.nih.gov/

Move confirmed-novel slides into `stimage_novel.csv` before running Part 2.

---
## Part 2 — Conversion: STimage-1K4M → HEST Format

**Input per sample** (downloaded from HuggingFace `jiawennnn/STimage-1K4M`):

| File | Notes |
|---|---|
| `Visium/image/{slide}.png` | H&E image — may be JPEG/TIFF despite `.png` extension; PIL auto-detects |
| `Visium/coord/{slide}_coord.csv` | Columns: barcode (index), `xaxis`, `yaxis`, `r` (spot radius px) |
| `Visium/gene_exp/{slide}_count.csv` | Dense matrix spots × genes — converted to sparse on read |
| `annotation/{slide}.csv` *(optional)* | Per-spot epithelium/stroma labels — merged into `adata.obs` if present |

**Pixel size** is computed from the spot radius column: `pixel_size = 27.5 µm / r_pixels`  
(Visium spots are 55 µm diameter; no external metadata file needed)

**Per-sample output** written to `hest_output/{slide}/`:

| File | Description |
|---|---|
| `aligned_adata.h5ad` | AnnData: sparse expression matrix, spot pixel coords, QC metrics, thumbnail |
| `aligned_fullres_HE.tif` | Pyramidal TIFF written by `HESTData.save()` |
| `metrics.json` | Pixel size, tissue, spot count, image dimensions |
| `patches.h5` | 256 × 256 patches at 0.5 µm/px, one per spot, barcoded to `adata.obs` |
| `patches_vis.jpg` | Patch location visualization overlay |

**Storage:** ~870 MB/sample → ~155 GB for 179 samples (patches dominate at 40–103 GB)

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────

def build_adata(coord_path: Path, count_path: Path, img: Image.Image):
    """
    Build a HEST-compatible AnnData from 1K4M coord + count CSVs.
    Returns (adata, coord_df).
    """
    coord = pd.read_csv(coord_path, index_col=0)

    # Dense CSV → sparse CSR immediately to avoid memory blowup
    # (~5000 spots × 20000 genes = 100M float32 values in a naive DataFrame)
    count = pd.read_csv(count_path, index_col=0)
    X = sp.csr_matrix(count.values.astype(np.float32))

    adata = sc.AnnData(X=X)
    adata.obs_names = count.index.tolist()
    adata.var_names = count.columns.tolist()

    # Align coord rows to count obs (same barcode order)
    coord = coord.reindex(adata.obs_names)

    # HEST convention: obsm['spatial'] = (n_spots, 2) as [x_pixel, y_pixel]
    adata.obsm['spatial'] = coord[['xaxis', 'yaxis']].values.astype(np.float64)

    # Downscaled thumbnail required by HESTData._verify_format
    scale = 0.05
    thumb = img.resize(
        (max(1, int(img.width * scale)), max(1, int(img.height * scale))),
        Image.LANCZOS,
    )
    adata.uns['spatial'] = {
        'ST': {
            'images': {'downscaled_fullres': np.asarray(thumb)},
            'scalefactors': {'tissue_downscaled_fullres_scalef': scale},
        }
    }

    return adata, coord


def compute_pixel_size(coord: pd.DataFrame) -> float:
    """Pixel size in µm/px from Visium spot radius (55 µm diameter → 27.5 µm radius)."""
    return VISIUM_SPOT_RADIUS_UM / coord['r'].median()


def load_annotation(slide: str, scratch_dir: Path):
    """Download per-spot annotation CSV from 1K4M (epithelium/stroma) if it exists."""
    try:
        local = hf_hub_download(
            repo_id=HF_STIMAGE_REPO,
            repo_type='dataset',
            filename=f'annotation/{slide}.csv',
            local_dir=str(scratch_dir / '_hf'),
            local_dir_use_symlinks=False,
        )
        return pd.read_csv(local, index_col=0)
    except Exception:
        return None   # absent for most slides — not required


print('Helper functions defined.')

In [ ]:
def convert_sample(row: pd.Series) -> bool:
    """
    Download one STimage-1K4M Visium sample from HuggingFace, convert to HEST
    format, and save to HEST_OUT/{slide}/.  Returns True on success.
    """
    slide   = row['slide']
    out_dir = HEST_OUT / slide

    if (out_dir / 'patches.h5').exists():
        print(f'  [skip] {slide}  (already converted)')
        return True

    scratch = SCRATCH / slide
    scratch.mkdir(parents=True, exist_ok=True)

    try:
        # ── 1. Download coord CSV, count CSV, image from HuggingFace ─────
        hf_files = {
            f'Visium/image/{slide}.png'          : scratch / 'image.src',
            f'Visium/coord/{slide}_coord.csv'    : scratch / 'coord.csv',
            f'Visium/gene_exp/{slide}_count.csv' : scratch / 'count.csv',
        }
        for hf_path, dest in hf_files.items():
            if not dest.exists():
                local = hf_hub_download(
                    repo_id=HF_STIMAGE_REPO,
                    repo_type='dataset',
                    filename=hf_path,
                    local_dir=str(scratch / '_hf'),
                    local_dir_use_symlinks=False,
                )
                shutil.copy(local, dest)
        print(f'  [downloaded]  {slide}')

        # ── 2. Load image ─────────────────────────────────────────────────
        # PIL.Image.open auto-detects format: 1K4M images are named .png but
        # may actually be JPEG or TIFF — do NOT assume PNG.
        img     = Image.open(scratch / 'image.src').convert('RGB')
        img_arr = np.asarray(img)

        # ── 3. Build AnnData from coord + count CSVs ──────────────────────
        adata, coord = build_adata(scratch / 'coord.csv', scratch / 'count.csv', img)

        # ── 4. Attach epithelium/stroma annotation if available ───────────
        ann = load_annotation(slide, scratch)
        if ann is not None:
            adata.obs = adata.obs.join(ann, how='left')
            print(f'    annotation merged  ({ann.shape[1]} columns)')

        # ── 5. Compute pixel size from spot radius ────────────────────────
        pixel_size = compute_pixel_size(coord)
        print(f'  [built]       {slide}  —  {adata.n_obs} spots · {adata.n_vars} genes · {pixel_size:.4f} µm/px')

        # ── 6. Metadata dict ──────────────────────────────────────────────
        meta = {
            'id'                     : slide,
            'st_technology'          : 'Visium',
            'species'                : 'Homo sapiens',
            'tissue'                 : str(row.get('tissue', '')),
            'disease_comment'        : str(row.get('title', '')),
            'pixel_size_um_estimated': round(pixel_size, 4),
            'spots_under_tissue'     : int(adata.n_obs),
            'adata_nb_col'           : int(adata.n_vars),
            'fullres_px_width'       : img.width,
            'fullres_px_height'      : img.height,
            'source'                 : 'STimage-1K4M',
            'pmid'                   : str(row.get('pmid', '')),
        }

        # ── 7. Wrap in HESTData ───────────────────────────────────────────
        # Pass numpy array — avoids any OpenSlide/cucim format dependency on
        # the source image; HESTData.save() writes the pyramidal TIFF itself.
        st = HESTData(adata=adata, img=img_arr, pixel_size=pixel_size, meta=meta)

        # ── 8. Tissue segmentation (required before dump_patches) ─────────
        st.segment_tissue(method='otsu')

        # ── 9. Save: aligned_adata.h5ad + aligned_fullres_HE.tif + metrics.json
        out_dir.mkdir(parents=True, exist_ok=True)
        st.save(str(out_dir), save_img=True, pyramidal=True)
        print(f'  [saved]       {slide}')

        # ── 10. Extract patches: 256×256 at 0.5 µm/px, one per spot ──────
        st.dump_patches(
            str(out_dir),
            name='patches',
            target_patch_size=PATCH_SIZE,
            target_pixel_size=TARGET_PIX_SIZE,
            dump_visualization=True,
        )
        print(f'  [patched]     {slide}  →  {out_dir}/patches.h5')
        return True

    except Exception as e:
        print(f'  [ERROR]  {slide}: {e}')
        return False

    finally:
        shutil.rmtree(scratch, ignore_errors=True)


print('convert_sample() defined.')

In [ ]:
# ── Load the novel candidates list ───────────────────────────────────────
novel = pd.read_csv(PIPELINE_OUT / 'stimage_novel.csv')
print(f'Novel samples to convert : {len(novel)}')
print(f'Tissue breakdown:\n{novel.tissue.value_counts().to_string()}')

# ── Run conversion ────────────────────────────────────────────────────────
# Processes one sample at a time: download (~200 MB) → convert → save → delete scratch
# Interrupt-safe: already-converted samples are skipped on resume.
success, failed = [], []

for i, (_, row) in enumerate(novel.iterrows()):
    print(f'\n[{i+1}/{len(novel)}] {row.slide}')
    if convert_sample(row):
        success.append(row['slide'])
    else:
        failed.append(row['slide'])

print(f'\n{"="*55}')
print(f'Conversion complete')
print(f'  Success : {len(success)}')
print(f'  Failed  : {len(failed)}')
if failed:
    print(f'  Failed slides: {failed}')

---
## Part 3 — Upload to HuggingFace

Converted samples are uploaded incrementally to `MahmoodLab/hest` (or a personal fork).  
Runs sample-by-sample — safe to interrupt and resume.

**Set your token before running:**
```bash
export HF_TOKEN=hf_...
```

**Storage estimate for 179 samples:**

| Component | Total |
|---|---|
| `aligned_fullres_HE.tif` (pyramidal) | ~45 GB |
| `patches.h5` | ~40–103 GB |
| `aligned_adata.h5ad` | ~6 GB |
| `metrics.json`, `patches_vis.jpg`, etc. | ~2 GB |
| **Total** | **~100–155 GB** |

In [ ]:
HF_TOKEN    = os.environ.get('HF_TOKEN')
TARGET_REPO = HF_HEST_REPO   # 'MahmoodLab/hest' or a personal fork

api = HfApi(token=HF_TOKEN)

def upload_sample(slide_id: str) -> None:
    """Upload all output files for one converted sample to HuggingFace."""
    sample_dir = HEST_OUT / slide_id
    if not sample_dir.exists():
        print(f'  [skip] {slide_id} — output dir not found')
        return
    for fpath in sorted(sample_dir.rglob('*')):
        if fpath.is_file():
            repo_path = f'data/{slide_id}/{fpath.name}'
            api.upload_file(
                path_or_fileobj=str(fpath),
                path_in_repo=repo_path,
                repo_id=TARGET_REPO,
                repo_type='dataset',
            )
    print(f'  [uploaded] {slide_id}')

for slide_id in success:
    upload_sample(slide_id)

print('All uploads complete.')

---
## Summary + Next Steps

**Per converted sample this pipeline produces:**

| File | Contents |
|---|---|
| `aligned_adata.h5ad` | Sparse expression matrix, spot pixel coords, QC metrics, thumbnail |
| `aligned_fullres_HE.tif` | Full-resolution pyramidal TIFF |
| `metrics.json` | Pixel size, tissue, spot count, image dimensions, source |
| `patches.h5` | 256 × 256 H&E patches at 0.5 µm/px, barcoded to `adata.obs` |
| `patches_vis.jpg` | Patch location visualization overlay |

**Scale:** ~179 samples · ~10K patches/sample · ~1.83M patches total · ~100–155 GB

---

**Next technologies to add (Matthew's slides 47):**
1. **Xenium** — imaging-based, single-cell, ~400 transcripts
2. **IMC** (Imaging Mass Cytometry) — spatial proteomics, ~50 proteins

**Open research question (Matthew's slide 31):**  
*"How much context should we give?"* — consider configurable patch sizes (512, 1024 px) for models that benefit from broader spatial context, as suggested by NOETIK/TARIO-2 scaling results.